In [1]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor

# =========================
# 0. 工具函数
# =========================

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def save_submission(filename, test_id, preds):
    submission = pd.DataFrame({
        "Id": test_id,
        "SalePrice": np.maximum(np.expm1(preds), 0)
    })
    path = f"/kaggle/working/{filename}"
    submission.to_csv(path, index=False)
    print(f"{filename} 已生成")
    return submission

# =========================
# 1. 读取数据
# =========================

data_path = Path("/kaggle/input/competitions/house-prices-advanced-regression-techniques")

train = pd.read_csv(data_path / "train.csv")
test = pd.read_csv(data_path / "test.csv")

print("训练集:", train.shape)
print("测试集:", test.shape)

test_id = test["Id"]
y = np.log1p(train["SalePrice"])

train_features = train.drop(["Id", "SalePrice"], axis=1)
test_features = test.drop(["Id"], axis=1)

all_features = pd.concat([train_features, test_features], axis=0)
print("合并后特征:", all_features.shape)

# =========================
# 2. 特征工程
# =========================

# 总面积
all_features["TotalSF"] = (
    all_features["TotalBsmtSF"].fillna(0)
    + all_features["1stFlrSF"].fillna(0)
    + all_features["2ndFlrSF"].fillna(0)
)

# 总卫生间数
all_features["TotalBathrooms"] = (
    all_features["FullBath"].fillna(0)
    + 0.5 * all_features["HalfBath"].fillna(0)
    + all_features["BsmtFullBath"].fillna(0)
    + 0.5 * all_features["BsmtHalfBath"].fillna(0)
)

# 总门廊面积
all_features["TotalPorchSF"] = (
    all_features["OpenPorchSF"].fillna(0)
    + all_features["EnclosedPorch"].fillna(0)
    + all_features["3SsnPorch"].fillna(0)
    + all_features["ScreenPorch"].fillna(0)
)

# 年龄类特征
all_features["HouseAge"] = all_features["YrSold"] - all_features["YearBuilt"]
all_features["RemodAge"] = all_features["YrSold"] - all_features["YearRemodAdd"]
all_features["GarageAge"] = all_features["YrSold"] - all_features["GarageYrBlt"]

# 是否类特征
all_features["HasBasement"] = (all_features["TotalBsmtSF"].fillna(0) > 0).astype(int)
all_features["HasGarage"] = (all_features["GarageArea"].fillna(0) > 0).astype(int)
all_features["HasFireplace"] = (all_features["Fireplaces"].fillna(0) > 0).astype(int)
all_features["HasPool"] = (all_features["PoolArea"].fillna(0) > 0).astype(int)
all_features["Has2ndFloor"] = (all_features["2ndFlrSF"].fillna(0) > 0).astype(int)

# 面积质量交互
all_features["OverallQual_TotalSF"] = all_features["OverallQual"] * all_features["TotalSF"]
all_features["OverallQual_GrLivArea"] = all_features["OverallQual"] * all_features["GrLivArea"]
all_features["GarageCars_GarageArea"] = all_features["GarageCars"].fillna(0) * all_features["GarageArea"].fillna(0)

# =========================
# 3. 缺失值处理 + 类别编码
# =========================

num_cols = all_features.select_dtypes(exclude="object").columns
cat_cols = all_features.select_dtypes(include="object").columns

for col in num_cols:
    all_features[col] = all_features[col].fillna(all_features[col].median())

for col in cat_cols:
    all_features[col] = all_features[col].fillna("None")
    encoder = LabelEncoder()
    all_features[col] = encoder.fit_transform(all_features[col].astype(str))

print("处理后特征:", all_features.shape)

n_train = train.shape[0]

X = all_features.iloc[:n_train, :].copy()
X_test = all_features.iloc[n_train:, :].copy()

print("X:", X.shape)
print("X_test:", X_test.shape)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 存每个模型的预测结果
model_oof = {}
model_test = {}

# =========================
# 4. RandomForest
# =========================

print("\n========== RandomForest ==========")

oof_rf = np.zeros(X.shape[0])
test_rf = np.zeros(X_test.shape[0])

for fold, (train_idx, valid_idx) in enumerate(kf.split(X), 1):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
    
    rf = RandomForestRegressor(
        n_estimators=500,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    )
    
    rf.fit(X_train, y_train)
    
    valid_pred = rf.predict(X_valid)
    test_pred = rf.predict(X_test)
    
    oof_rf[valid_idx] = valid_pred
    test_rf += test_pred / kf.n_splits
    
    print(f"Fold {fold} RF RMSE:", rmse(y_valid, valid_pred))

rf_cv = rmse(y, oof_rf)
print("RandomForest 整体 CV RMSE:", rf_cv)

model_oof["rf"] = oof_rf
model_test["rf"] = test_rf

save_submission("submission_rf.csv", test_id, test_rf)

# =========================
# 5. XGBoost
# =========================

print("\n========== XGBoost ==========")

try:
    from xgboost import XGBRegressor
    
    oof_xgb = np.zeros(X.shape[0])
    test_xgb = np.zeros(X_test.shape[0])
    
    for fold, (train_idx, valid_idx) in enumerate(kf.split(X), 1):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
        
        xgb = XGBRegressor(
            n_estimators=1200,
            learning_rate=0.02,
            max_depth=3,
            min_child_weight=2,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=1.5,
            objective="reg:squarederror",
            eval_metric="rmse",
            random_state=42,
            n_jobs=-1
        )
        
        xgb.fit(X_train, y_train)
        
        valid_pred = xgb.predict(X_valid)
        test_pred = xgb.predict(X_test)
        
        oof_xgb[valid_idx] = valid_pred
        test_xgb += test_pred / kf.n_splits
        
        print(f"Fold {fold} XGB RMSE:", rmse(y_valid, valid_pred))
    
    xgb_cv = rmse(y, oof_xgb)
    print("XGBoost 整体 CV RMSE:", xgb_cv)
    
    model_oof["xgb"] = oof_xgb
    model_test["xgb"] = test_xgb
    
    save_submission("submission_xgb.csv", test_id, test_xgb)

except Exception as e:
    print("XGBoost 跑不了，原因:", e)

# =========================
# 6. LightGBM
# =========================

print("\n========== LightGBM ==========")

try:
    from lightgbm import LGBMRegressor
    
    oof_lgbm = np.zeros(X.shape[0])
    test_lgbm = np.zeros(X_test.shape[0])
    
    for fold, (train_idx, valid_idx) in enumerate(kf.split(X), 1):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
        
        lgbm = LGBMRegressor(
            n_estimators=1500,
            learning_rate=0.02,
            max_depth=4,
            num_leaves=20,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.05,
            reg_lambda=0.5,
            random_state=42,
            n_jobs=-1,
            verbose=-1
        )
        
        lgbm.fit(X_train, y_train)
        
        valid_pred = lgbm.predict(X_valid)
        test_pred = lgbm.predict(X_test)
        
        oof_lgbm[valid_idx] = valid_pred
        test_lgbm += test_pred / kf.n_splits
        
        print(f"Fold {fold} LGBM RMSE:", rmse(y_valid, valid_pred))
    
    lgbm_cv = rmse(y, oof_lgbm)
    print("LightGBM 整体 CV RMSE:", lgbm_cv)
    
    model_oof["lgbm"] = oof_lgbm
    model_test["lgbm"] = test_lgbm
    
    save_submission("submission_lgbm_v2.csv", test_id, test_lgbm)

except Exception as e:
    print("LightGBM 跑不了，原因:", e)

# =========================
# 7. 模型融合
# =========================

print("\n========== 模型融合 ==========")

print("可用模型:", list(model_oof.keys()))

# 如果三个模型都有，就用推荐权重
if set(["rf", "xgb", "lgbm"]).issubset(model_oof.keys()):
    blend_oof = (
        0.10 * model_oof["rf"]
        + 0.35 * model_oof["xgb"]
        + 0.55 * model_oof["lgbm"]
    )
    
    blend_test = (
        0.10 * model_test["rf"]
        + 0.35 * model_test["xgb"]
        + 0.55 * model_test["lgbm"]
    )

# 如果只有 XGB 和 LGBM，就两个融合
elif set(["xgb", "lgbm"]).issubset(model_oof.keys()):
    blend_oof = (
        0.40 * model_oof["xgb"]
        + 0.60 * model_oof["lgbm"]
    )
    
    blend_test = (
        0.40 * model_test["xgb"]
        + 0.60 * model_test["lgbm"]
    )

# 其他情况：取已有模型平均
else:
    blend_oof = np.mean([model_oof[name] for name in model_oof.keys()], axis=0)
    blend_test = np.mean([model_test[name] for name in model_test.keys()], axis=0)

blend_cv = rmse(y, blend_oof)

print("融合模型整体 CV RMSE:", blend_cv)

save_submission("submission_blend.csv", test_id, blend_test)

# =========================
# 8. 汇总结果
# =========================

print("\n========== CV 结果汇总 ==========")

if "rf" in model_oof:
    print("RandomForest:", rmse(y, model_oof["rf"]))

if "xgb" in model_oof:
    print("XGBoost:", rmse(y, model_oof["xgb"]))

if "lgbm" in model_oof:
    print("LightGBM:", rmse(y, model_oof["lgbm"]))

print("Blend:", blend_cv)

print("\n生成的文件在 /kaggle/working 里面：")
print("submission_rf.csv")
print("submission_xgb.csv")
print("submission_lgbm_v2.csv")
print("submission_blend.csv")

训练集: (1460, 81)
测试集: (1459, 80)
合并后特征: (2919, 79)
处理后特征: (2919, 93)
X: (1460, 93)
X_test: (1459, 93)

========== RandomForest ==========
Fold 1 RF RMSE: 0.1485140821862141
Fold 2 RF RMSE: 0.12228181801736689
Fold 3 RF RMSE: 0.16106735806000544
Fold 4 RF RMSE: 0.14235778590786674
Fold 5 RF RMSE: 0.1143297912350666
RandomForest 整体 CV RMSE: 0.138772482740722
submission_rf.csv 已生成

========== XGBoost ==========
Fold 1 XGB RMSE: 0.1304512902939164
Fold 2 XGB RMSE: 0.11162467558623962
Fold 3 XGB RMSE: 0.1613704564309041
Fold 4 XGB RMSE: 0.1178707836985489
Fold 5 XGB RMSE: 0.1065524849184937
XGBoost 整体 CV RMSE: 0.12709444145187088
submission_xgb.csv 已生成

========== LightGBM ==========
Fold 1 LGBM RMSE: 0.13834426964564836
Fold 2 LGBM RMSE: 0.11656554483393754
Fold 3 LGBM RMSE: 0.1609186203189083
Fold 4 LGBM RMSE: 0.1277104075110512
Fold 5 LGBM RMSE: 0.10959900771981446
LightGBM 整体 CV RMSE: 0.13186611106070295
submission_lgbm_v2.csv 已生成

========== 模型融合 ==========
可用模型: ['rf', 'xgb', 'lgbm']
融